# Skill — 封裝成可複用的能力包

本筆記本示範如何將 Agent 的「專業能力」封裝成 **Skill**，並整合到 LangGraph Supervisor 中。

### Tool vs Skill

| 概念 | 定義 | 例子 |
|------|------|------|
| **Tool** | 單一函式，完成一個動作 | `check_warranty(serial)` |
| **Skill** | Tool + Prompt + Workflow + Examples | HP 維修診斷專家 |

### Skill 的四個組成元素

1. **System Prompt** — 定義角色與行為邊界
2. **Tools** — 這個 Skill 可以使用的工具集
3. **Workflow** — 處理流程（分類 → 診斷 → 引導）
4. **Examples** — 少樣本示範，確保輸出格式一致

> **核心概念**：Supervisor 中的每個 Worker 本質上就是一個 Skill。Skill 讓專業能力可以被版本化、測試、重複使用。

In [ ]:
%%capture
!pip install langchain langchain-openai langgraph pydantic

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""

---
## 1. 定義 Skill 的資料結構

我們用 Pydantic 定義 Skill 的結構，讓每個 Skill 的組成清晰可驗證。

In [ ]:
from dataclasses import dataclass, field
from typing import Callable, Any
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

@dataclass
class Skill:
    """可複用的 Agent 能力單元"""
    name: str
    description: str          # 給 Supervisor 看的簡短說明
    system_prompt: str        # 定義角色與行為邊界
    tools: list = field(default_factory=list)    # 這個 Skill 可用的工具
    examples: list = field(default_factory=list) # 少樣本示範

    def invoke(self, query: str, llm: Any) -> str:
        """執行此 Skill 處理用戶問題"""
        from langgraph.prebuilt import create_react_agent

        messages = []
        # 加入少樣本示範
        for ex in self.examples:
            messages.append(HumanMessage(content=ex["input"]))
            from langchain_core.messages import AIMessage
            messages.append(AIMessage(content=ex["output"]))
        messages.append(HumanMessage(content=query))

        if self.tools:
            agent = create_react_agent(llm, self.tools,
                                       state_modifier=self.system_prompt)
            result = agent.invoke({"messages": messages})
            return result["messages"][-1].content
        else:
            resp = llm.invoke([SystemMessage(content=self.system_prompt)] + messages)
            return resp.content

print("✅ Skill 資料結構定義完成")

---
## 2. 實作兩個 HP Skill

### Skill 1：HP 產品規格專家
- **Tools**：`search_specs`, `compare_models`
- **知識**：HP 命名規則（EliteBook 8xx = 商務旗艦）

### Skill 2：HP 維修診斷專家
- **Tools**：`check_warranty`, `get_repair_sop`
- **Workflow**：問題分類 → 保固確認 → SOP 引導

In [ ]:
# ── 定義工具 ──────────────────────────────────────────

@tool
def search_specs(model: str) -> str:
    """搜尋 HP 產品規格"""
    specs = {
        "HP EliteBook 840 G11": "CPU: Intel Core Ultra 7 165U | RAM: 32GB | 顯示: 14吋 2.8K OLED | 重量: 1.37kg | 電池: 51Wh",
        "HP ZBook Studio G10":  "CPU: Intel Core i9-13900H | RAM: 64GB | GPU: NVIDIA RTX 4070 | 顯示: 16吋 OLED | 重量: 1.79kg",
        "HP LaserJet Pro 4001n": "列印速度: 40ppm | 解析度: 1200dpi | 連接: USB/網路 | 月印量: 10,000頁",
    }
    for key, val in specs.items():
        if model.lower() in key.lower():
            return f"{key}: {val}"
    return f"查無 {model} 的規格資料"

@tool
def compare_models(model_a: str, model_b: str) -> str:
    """比較兩款 HP 產品"""
    return f"[比較結果] {model_a} vs {model_b}：EliteBook 適合一般商務用戶，ZBook 適合需要 GPU 的工程師/設計師"

@tool
def check_warranty(serial_number: str) -> str:
    """確認 HP 產品保固狀態"""
    db = {"SN12345": "有效（到期：2026-12-31）", "SN99999": "已過期（到期：2024-01-01）"}
    return db.get(serial_number, "查無此序號，請確認後再試")

@tool
def get_repair_sop(issue: str) -> str:
    """取得 HP 維修 SOP"""
    sops = {
        "無法開機": "1. 確認電源線 2. 嘗試硬重啟（長按電源10秒）3. 進 BIOS 診斷 4. 聯絡技術支援",
        "卡紙":     "1. 關閉電源 2. 打開前蓋 3. 輕輕取出紙張 4. 重新對齊後重啟",
        "螢幕": "1. 調整亮度設定 2. 更新顯示驅動 3. 測試外接螢幕 4. 送修",
    }
    for key, sop in sops.items():
        if key in issue:
            return sop
    return "請聯絡 HP 技術支援：0800-012-345（週一至週五 9:00-18:00）"

print("✅ 工具定義完成")

In [ ]:
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# ── Skill 1：HP 產品規格專家 ──────────────────────────
spec_skill = Skill(
    name="hp_spec_expert",
    description="回答 HP 產品規格、型號比較、選購建議",
    system_prompt="""你是 HP 產品規格專家。
HP 命名規則：
- EliteBook 8xx = 商務旗艦（輕薄、長效電池）
- ZBook = 行動工作站（高效能 GPU，設計/工程用）
- ProBook = 商務入門（CP 值優先）
- LaserJet Pro = 中小企業雷射印表機

回答時請先確認用戶需求，再推薦最適合的產品。""",
    tools=[search_specs, compare_models],
    examples=[
        {"input": "EliteBook 適合什麼用途？",
         "output": "EliteBook 系列專為商務用戶設計，主打輕薄、長效電池與企業安全功能（如 HP Wolf Security），適合需要長時間外出辦公的商務人士。"},
    ]
)

# ── Skill 2：HP 維修診斷專家 ──────────────────────────
repair_skill = Skill(
    name="hp_repair_expert",
    description="處理 HP 產品故障診斷、保固查詢、維修引導",
    system_prompt="""你是 HP 維修診斷專家，處理流程：
1. 問題分類：硬體故障 / 軟體問題 / 消耗品
2. 保固確認：若用戶提供序號，先查保固狀態
3. SOP 引導：提供逐步維修指引
4. 升級處理：若 SOP 無法解決，引導聯絡技術支援

語氣：親切、專業、耐心，避免使用過多技術術語。""",
    tools=[check_warranty, get_repair_sop],
    examples=[
        {"input": "我的印表機卡紙了",
         "output": "我來幫您處理卡紙問題。首先請確認您的型號，然後我會提供逐步的排除步驟。"},
    ]
)

print("✅ 兩個 Skill 建立完成：")
print(f"  - {spec_skill.name}: {spec_skill.description}")
print(f"  - {repair_skill.name}: {repair_skill.description}")

In [ ]:
# 直接測試各個 Skill
print("=== 測試 HP 產品規格專家 ===")
result = spec_skill.invoke("EliteBook 840 G11 和 ZBook Studio G10 哪個適合我？我是平面設計師。", llm)
print(result)
print()
print("=== 測試 HP 維修診斷專家 ===")
result = repair_skill.invoke("我的序號 SN12345，筆電無法開機，請問怎麼辦？", llm)
print(result)

---
## 3. Skill Registry — 集中管理所有 Skill

SkillRegistry 讓 Supervisor 能夠根據問題類型自動選擇合適的 Skill。

In [ ]:
class SkillRegistry:
    """集中管理所有 Skill，提供查詢與路由"""

    def __init__(self):
        self._skills: dict[str, Skill] = {}

    def register(self, skill: Skill):
        self._skills[skill.name] = skill
        return self

    def list_skills(self) -> str:
        """產生供 Supervisor LLM 選擇的技能清單"""
        lines = ["可用 Skill："]
        for name, skill in self._skills.items():
            lines.append(f"  - {name}: {skill.description}")
        return "\n".join(lines)

    def get(self, name: str) -> Skill:
        return self._skills[name]

# 建立 Registry 並註冊所有 Skill
registry = SkillRegistry()
registry.register(spec_skill).register(repair_skill)

print(registry.list_skills())

---
## 4. Supervisor + Skill 整合

Supervisor 接收用戶問題後，選擇最適合的 Skill 來處理。這就是 Supervisor 模式的精髓：**分工協作**。

```
User Query
     │
     ▼
Supervisor (路由決策)
     ├─── 規格/選購問題 ──► hp_spec_expert Skill
     └─── 故障/維修問題 ──► hp_repair_expert Skill
               │
               ▼
         最終回答給用戶
```

In [ ]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import SystemMessage, HumanMessage
import operator

class AgentState(TypedDict):
    messages: Annotated[list, operator.add]
    selected_skill: str
    final_answer: str

# ── Supervisor 節點：決定使用哪個 Skill ──
def supervisor_node(state: AgentState) -> AgentState:
    user_query = state["messages"][-1].content

    routing_prompt = f"""根據用戶問題，選擇最適合的 Skill 名稱（只回答 Skill 名稱，不要其他文字）。

{registry.list_skills()}

用戶問題：{user_query}

選擇的 Skill 名稱："""

    response = llm.invoke([HumanMessage(content=routing_prompt)])
    selected = response.content.strip()

    # 找到最接近的 Skill 名稱
    for skill_name in ["hp_spec_expert", "hp_repair_expert"]:
        if skill_name in selected:
            selected = skill_name
            break
    else:
        selected = "hp_spec_expert"  # 預設

    print(f"  [Supervisor] 路由到: {selected}")
    return {"selected_skill": selected}

# ── Worker 節點：執行選定的 Skill ──
def worker_node(state: AgentState) -> AgentState:
    skill = registry.get(state["selected_skill"])
    user_query = state["messages"][-1].content
    answer = skill.invoke(user_query, llm)
    return {"final_answer": answer}

# ── 路由函數 ──
def route_to_skill(state: AgentState) -> Literal["worker"]:
    return "worker"

# ── 建立 Graph ──
graph = StateGraph(AgentState)
graph.add_node("supervisor", supervisor_node)
graph.add_node("worker", worker_node)

graph.add_edge(START, "supervisor")
graph.add_edge("supervisor", "worker")
graph.add_edge("worker", END)

app = graph.compile()
print("✅ Supervisor + Skill Graph 建立完成")

In [ ]:
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
# 測試 Supervisor + Skill
def ask(query: str):
    print(f"\n問題：{query}")
    print("-" * 50)
    result = app.invoke({
        "messages": [HumanMessage(content=query)],
        "selected_skill": "",
        "final_answer": "",
    })
    print(f"回答：{result['final_answer']}")

ask("HP EliteBook 840 G11 的重量和電池容量是多少？")
ask("我的序號 SN99999，筆電螢幕突然變黑了，保固還有效嗎？")
ask("EliteBook 和 ZBook 的主要差異是什麼？我需要跑 3D 渲染。")

---
## 5. Skill 版本化管理

Skill 的一大優點是**可版本化**：不同版本的 Skill 可以並存，方便 A/B 測試和逐步升級。

In [ ]:
from dataclasses import dataclass

@dataclass
class VersionedSkill(Skill):
    version: str = "1.0.0"
    changelog: str = ""

# v1.0 — 基礎版
spec_skill_v1 = VersionedSkill(
    name="hp_spec_expert",
    description="回答 HP 產品規格",
    system_prompt="你是 HP 產品專家，根據用戶需求提供產品規格資訊。",
    tools=[search_specs],
    version="1.0.0",
    changelog="初始版本"
)

# v1.1 — 加入比較功能 + 更豐富的 Prompt
spec_skill_v1_1 = VersionedSkill(
    name="hp_spec_expert",
    description="回答 HP 產品規格、型號比較、選購建議",
    system_prompt="""你是 HP 資深產品顧問。
HP 命名規則：EliteBook（商務）/ ZBook（工作站）/ ProBook（入門）
提供建議時先詢問使用情境，再匹配最適合的產品線。""",
    tools=[search_specs, compare_models],
    version="1.1.0",
    changelog="新增 compare_models 工具；改善 Prompt 以提供更精準的選購建議"
)

print(f"v1.0.0 工具數：{len(spec_skill_v1.tools)}")
print(f"v1.1.0 工具數：{len(spec_skill_v1_1.tools)}")
print(f"v1.1.0 更新說明：{spec_skill_v1_1.changelog}")
print()
print("💡 部署新版本只需在 SkillRegistry 中替換，Supervisor 不需改動")

---
## 總結

| 特性 | 說明 |
|------|------|
| **封裝性** | Prompt + Tools + Workflow 打包成一個單元 |
| **可複用** | 同一個 Skill 可被多個 Supervisor 使用 |
| **可版本化** | 獨立更新，不影響其他 Skill 或 Supervisor |
| **可測試** | 每個 Skill 可以獨立進行單元測試 |
| **可組合** | Supervisor 動態選擇 Skill 組合解決複雜問題 |

### MCP vs Skill

```
MCP Server  ──►  提供工具（Tool 層）
    ↓
Skill       ──►  封裝能力（Tool + Prompt + Workflow）
    ↓
Supervisor  ──►  協調多個 Skill（決策層）
```

三者互補，共同構成企業 AI Agent 的完整架構。